### RESUMO
I. Funções
Blocos de código reutilizáveis que recebem inputs e retornam outputs
Analogia: Como uma receita de bolo - você define os ingredientes (parâmetros) e obtém o resultado

II. Argumentos Posicionais vs Nomeados
Posicionais: Ordem importa. Nomeados: Especifica o parâmetro explicitamente
Analogia: Endereço - "Rua X, número Y" (posicional) vs "número Y da Rua X" (nomeado)

III. Namespaces e Escopo
Variáveis dentro da função são locais, fora são globais
Analogia: Sua casa (local) vs rua (global) - o que acontece na casa fica na casa

IV. Retorno Múltiplo
Na verdade retorna uma tupla que pode ser desempacotada
Analogia: Pacote com vários itens que você desembrulha separadamente

V. Funções como Objetos
Funções podem ser passadas como argumentos, armazenadas em listas
Analogia: Ferramentas em uma caixa - você escolhe qual usar quando precisar

VI. Lambda Functions
Funções anônimas de uma linha, úteis para operações simples
Analogia: Atalho de teclado vs programa completo

VII. Generators
Produzem valores sob demanda, economizando memória
Analogia: Torneira (água quando abre) vs balde cheio (toda água de uma vez)

VIII. Tratamento de Exceções
try/except para lidar com erros graciosamente
Analogia: Airbag - só ativa quando há colisão (erro)

### APLICAÇÃO REAL
I. Engenharia de Dados
Pipelines ETL: Funções para cada etapa (extract, transform, load)
Cloud Functions: Lambdas em AWS/Azure para processamento sob demanda
Validação de Dados: try/except para lidar com dados corrompidos
Processamento Streaming: Generators para dados que chegam continuamente

II. Análise de Dados
Limpeza de Dados: Listas de funções de transformação (como no exemplo dos estados)
Feature Engineering: Lambda functions em operações com DataFrames
Validação: Verificar tipos de dados e formatos antes do processamento

In [2]:
import re
from typing import List, Dict, Union, Generator
from datetime import datetime

# FUNÇÕES DE LIMPEZA (Namespace Global - reutilizáveis)
def remover_caracteres_especiais(texto: str) -> str:
    """Remove caracteres especiais usando regex"""
    return re.sub(r'[!@#$%^&*()]', '', texto)

def normalizar_email(texto: str) -> str:
    """Converte para minúsculas e remove espaços - assume que é email"""
    return texto.strip().lower()

def extrair_e_validar_id(texto: str) -> Union[str, None]:
    """Extrai IDs numéricos de 8 dígitos"""
    try:
        # Encontra sequências de 8 dígitos
        ids = re.findall(r'\d{8}', texto)
        return ids[0] if ids else None
    except (AttributeError, TypeError):
        return None

def classificar_tipo_dado(texto: str) -> str:
    """Classifica o tipo de dado baseado em padrões"""
    texto_limpo = texto.strip().lower()
    
    if re.match(r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$', texto_limpo):
        return 'email'
    elif re.match(r'^\d{8}$', texto_limpo):
        return 'id_produto'
    elif any(palavra in texto_limpo for palavra in ['nome', 'joão', 'maria', 'carlos']):
        return 'nome_usuario'
    else:
        return 'desconhecido'

# LISTA DE OPERAÇÕES GLOBAL (reutilizável)
OPS_LIMPEZA_GERAIS = [
    str.strip,
    remover_caracteres_especiais,
    lambda x: normalizar_email(x) if '@' in x else x,
    lambda x: x.title() if any(nome in x.lower() for nome in ['joão', 'maria', 'carlos']) else x
]

# PIPELINE PRINCIPAL CORRIGIDA
def pipeline_limpeza_dados(dados_brutos: List[str], 
                          operacoes: List[callable] = None) -> List[Dict]:
    """
    Pipeline robusto para limpar dados de usuários de e-commerce
    """
    
    # Usa operações padrão se nenhuma for fornecida
    if operacoes is None:
        operacoes = OPS_LIMPEZA_GERAIS
    
    # Generator CORRETO com tipo definido
    def processar_lote(dados: List[str]) -> Generator[str, None, None]:
        """Generator para processamento lazy com tratamento de erro"""
        for linha in dados:
            try:
                if not linha or not isinstance(linha, str):
                    continue
                    
                # Aplica cadeia de operações
                dado_processado = linha
                for operacao in operacoes:
                    dado_processado = operacao(dado_processado)
                
                yield dado_processado
                
            except Exception as e:
                print(f"🚨 Erro processando '{linha}': {e}")
                continue  # Continua com próximo item
    
    # Processamento principal
    dados_processados = []
    for dado_limpo in processar_lote(dados_brutos):
        if dado_limpo and dado_limpo.strip():  # Filtra vazios
            dados_processados.append({
                'dado_original': next((d for d in dados_brutos if d.strip() == dado_limpo), 'N/A'),
                'dado_limpo': dado_limpo,
                'tipo': classificar_tipo_dado(dado_limpo),
                'timestamp': datetime.now().isoformat(),
                'comprimento_original': len(dado_limpo)
            })
    
    return dados_processados

# USO PRÁTICO CORRIGIDO
def demonstrar_pipeline():
    """Demonstra o funcionamento do pipeline"""
    
    dados_sujos = [
        "  JOÃO@EMAIL.COM  ",
        "maria!silva#gmail.com",
        "produto123",
        "  CARLOS SANTOS  ",
        "   ",  # string vazia
        "ID: 12345678",  # ID válido
        "2024-01-15"  # data
    ]
    
    print("=== PIPELINE DE LIMPEZA DE DADOS ===")
    print(f"📥 Input: {len(dados_sujos)} registros")
    
    # Método 1: Usando operações padrão
    resultado1 = pipeline_limpeza_dados(dados_sujos)
    print(f"📤 Output (padrão): {len(resultado1)} registros limpos")
    
    # Método 2: Operações customizadas
    ops_customizadas = [
        str.strip,
        lambda x: x.upper(),  # Força maiúsculas
        remover_caracteres_especiais
    ]
    
    resultado2 = pipeline_limpeza_dados(dados_sujos, ops_customizadas)
    print(f"📤 Output (customizado): {len(resultado2)} registros limpos")
    
    # Demonstrando resultados
    print("\n🔍 DETALHES DOS RESULTADOS:")
    for i, item in enumerate(resultado1[:3]):  # Mostra apenas 3 primeiros
        print(f"  {i+1}. Original: '{item['dado_original']}'")
        print(f"     Limpo: '{item['dado_limpo']}'")
        print(f"     Tipo: {item['tipo']}")
        print()

# EXECUÇÃO SEGURA COM TRY/EXCEPT
if __name__ == "__main__":
    try:
        demonstrar_pipeline()
        
        # Teste adicional com dados problemáticos
        print("=== TESTE COM DADOS PROBLEMÁTICOS ===")
        dados_problematicos = [
            None,  # None value
            123,   # Número
            "",    # String vazia
            "  TESTE@EMAIL.COM  "
        ]
        
        resultado_seguro = pipeline_limpeza_dados([str(x) if x else "" for x in dados_problematicos])
        print(f"✅ Processados com segurança: {len(resultado_seguro)} de {len(dados_problematicos)}")
        
    except Exception as e:
        print(f"❌ Erro na execução: {e}")

=== PIPELINE DE LIMPEZA DE DADOS ===
📥 Input: 7 registros
📤 Output (padrão): 6 registros limpos
📤 Output (customizado): 6 registros limpos

🔍 DETALHES DOS RESULTADOS:
  1. Original: 'N/A'
     Limpo: 'Joãoemail.Com'
     Tipo: nome_usuario

  2. Original: 'N/A'
     Limpo: 'Mariasilvagmail.Com'
     Tipo: nome_usuario

  3. Original: 'produto123'
     Limpo: 'produto123'
     Tipo: desconhecido

=== TESTE COM DADOS PROBLEMÁTICOS ===
✅ Processados com segurança: 2 de 4


In [1]:
"""
Exercício F1: Split Básico (Netflix 1%)
Objetivo: Dominar split() para dados estruturados.
"""

# Dados simples
usuarios = [
    "ana.silva@email.com,28,SP",
    "carlos.santos@email.com,35,RJ",
    "maria.oliveira@email.com,42,MG"
]

# Tarefa: Para cada string, divida pela vírgula e imprima:
# "Nome: [primeira parte antes do @], Idade: [segunda parte], Estado: [terceira parte]"

# Exemplo de saída:
# "Nome: ana.silva, Idade: 28, Estado: SP"

#Dica: Use for loop e split(','). Para extrair só o nome do email, use split('@')[0].


#Contar o numero de valores -> [-1] para comecar do zero
quantid = len(usuarios)
print(quantid)


#---------
print("Nome dos usuarios: ")

#Usa a quantidade de valores da lista como referencia
for x in range(0, quantid):

    #Corre por todos os valores da lista
    x1 = usuarios[x].split(',')

    #Exibe a primeira parte da virgula
    x2 = x1[0].split(',')

    #Exclui a parte depois do @
    x3 = x2[0].split('@')
    print(x3[0])

3
Nome dos usuarios: 
ana.silva
carlos.santos
maria.oliveira


In [2]:
"""
Exercício F2: Try/Except Básico (Netflix 10%)
Objetivo: Validar conversão de tipos sem quebrar o programa.
"""
# Dados com problemas
numeros = ["10", "25", "abc", "30", "dez", "45"]

# Tarefa: Some APENAS os números válidos, ignorando os inválidos.
# Imprima: "Soma dos números válidos: X"
# Imprima: "Itens inválidos encontrados: Y"

# Exemplo de saída:
# "Soma dos números válidos: 110"
# "Itens inválidos encontrados: 2"
#💡 Dica: Use try: int(item) e except ValueError:. #Mantenha contadores separados.

y = 0
cont_valid = 0
cont_invalid = 0

for x in numeros:
    try:
        #tentar tranaformar em num inteiro
        x_int = int(x)
        soma = y + x_int
        y = soma
        cont_valid += 1

    except ValueError:
        #Se a transformacao der erro
        cont_invalid += 1

print(f"Soma dos numeros validos: {y}")
print(f"Num validos: {cont_valid}")
print(f"Num invalidos: {cont_invalid}")

Soma dos numeros validos: 110
Num validos: 4
Num invalidos: 2


In [1]:
"""
Exercício F3: Função Simples (Netflix 20%)
Objetivo: Criar função modular com um propósito claro.

python
# Tarefa: Crie uma função formatar_nome que:
# 1. Recebe uma string (ex: "  joão DA silva  ")
# 2. Remove espaços extras (strip)
# 3. Converte para "João da Silva" (cada palavra capitalizada, exceto "da", "de", "dos")
# 4. Retorna a string formatada

# Teste com:
# nomes = ["  maria antonieta  ", "carlos DE oliveira", "  PEDRO  "]
💡 Dica: Use split(), capitalize(), e verifique se palavra está em ["da", "de", "dos"].
"""

def formatar_nome(nome):
    #remove espaços extras no inicio e no final
    nome = nome.strip()
    
    #divide numa lista de palavras
    palavras = nome.split()
    
    #lista para armazenar as palavras ja formatadas
    palavras_formatadas = []
    
    #iterage com cada valor da lista [palavras]
    for palavra in palavras:
        #converte o valor para minusculo
        palavra_lower = palavra.lower()
    
        #verifica se é palavra que deve ficar em minusculo
        if palavra_lower in ["da", "de", "dos"] and palavras_formatadas:
            palavras_formatadas.append(palavra_lower)
        else:
            palavras_formatadas.append(palavra.capitalize())
            
    #junta todas as palavras formatadas com um espaço entre elas    
    return " ".join(palavras_formatadas)
    
#teste com exemplos 
nomes = [" maria antonia ", "carlos d ", " PEDRO "]

print("Nomes formatados:")
for nome in nomes:
    formatado = formatar_nome(nome)
    print(f"{nome} -> {formatado}")
    

Nomes formatados:
 maria antonia  -> Maria Antonia
carlos d  -> Carlos D
 PEDRO  -> Pedro


In [1]:
# Tarefa: Transforme cada linha em um dicionário com chaves:
# {'usuario': '', 'acao': '', 'data': '', 'hora': ''}

# Separe data e hora (a string tem um espaço entre elas)
# Dica: Primeiro split(','), depois split(' ') na parte da data/hora.

logs = [
    "user001,login,2024-01-15 10:30:00",
    "user002,compra,2024-01-15 11:15:00",
    "user001,logout,2024-01-15 12:00:00"
]

print(len(logs))
n=len(logs)


for i in range (0,n):
    info_virg=logs[i].split(',')
    
    info_spc=[]
    
    for posicao in range(0,3):
        if posicao == 2:
            spc=info_virg[posicao].split(' ')
            info_virg.pop(posicao)
            info_virg.append(spc[0])
            info_virg.append(spc[1])
    
    dic={}
    dic["user:"] = info_virg[0]
    dic["acao:"] = info_virg[1]
    dic["data:"] = info_virg[2]
    dic["hora:"] = info_virg[3]
    
    print(dic.items())

3
dict_items([('user:', 'user001'), ('acao:', 'login'), ('data:', '2024-01-15'), ('hora:', '10:30:00')])
dict_items([('user:', 'user002'), ('acao:', 'compra'), ('data:', '2024-01-15'), ('hora:', '11:15:00')])
dict_items([('user:', 'user001'), ('acao:', 'logout'), ('data:', '2024-01-15'), ('hora:', '12:00:00')])


In [2]:
# Dados para validação
registros = [
    "USER001,Produto A,29.90,3",
    "USER002,Produto B,dez,2",    # preço inválido
    "USER003,,15.50,1",           # produto vazio
    "USER004,Produto C,-5.00,4",  # preço negativo
    "USER005,Produto D,10.00,0"   # quantidade zero
]

# Tarefa: Crie uma função validar_registro que:
# 1. Verifica se tem 4 campos
# 2. Verifica se usuário começa com "USER"
# 3. Verifica se produto não é vazio
# 4. Verifica se preço é número positivo
# 5. Verifica se quantidade é inteiro positivo

# Retorne (True/False, lista_de_erros)
#----------------------------------------

log = ["USRR002,Produto B,-10,2.5",
    "USER004,,5.00,4"
]

#Separando
NumElements = len(log)


for LogPosicao in range(0, NumElements):
    #Separando as virgulas
    LogSeparado = log[LogPosicao].split(',')
    print("--------------")
    print(LogSeparado)

    print("Descrição:")
    #Verificar o numero de elementos em cada valor no [LogSeparado]
    print(f"1. Numero de elementos: {len(LogSeparado)}")
    
    #Verificar se o primeiro valor contem [USER]
    PrimeiroValor = LogSeparado[0]
    ContemUser = PrimeiroValor[0:4]
    
    if ContemUser == "USER":
        print("2. Contem o valor [USER]")
    else:
        print("2. Não contem [USER]")
    
    #Verificar se o segundo valor está vazio ou não 
    SegundoValor = LogSeparado[1]
    
    if SegundoValor == "":
        print("3. O valor do PRODUTO esta VAZIO")
    else:
        print("3. O valor do PRODUTO NÃO esta VAZIO")
    
    try:
        #Verificar se o 3 valor é número e é positivo
        TerceiroValor = LogSeparado[2]
        
        if  float(LogSeparado[2]) > 0:
            print("4. PREÇO é um numero positivo")
        else:
            print("4. PREÇO é numero negativo")
            
    except ValueError:
        #Se o valor não for um número
        print("4. PREÇO não numero")
    
    except IndexError:
        #Caso o valor não exista
        print("4. Erro. Não encontrado na lista")
    
    #Se o ultimo valor é inteiro e positivo
    QuartoValor = LogSeparado[3]
    
    #>> Transformando em float para tirar do estado de "String"
    FloatQuartoValor = float(QuartoValor)
    
    
    #print(FloatQuartoValor.is_integer())
    if FloatQuartoValor.is_integer():
        print("5. A QUANTIDADE é um numero inteiro")
    else:
        print("5. A QUANTIDADE NÃO é um numero inteiro")

--------------
['USRR002', 'Produto B', '-10', '2.5']
Descrição:
1. Numero de elementos: 4
2. Não contem [USER]
3. O valor do PRODUTO NÃO esta VAZIO
4. PREÇO é numero negativo
5. A QUANTIDADE NÃO é um numero inteiro
--------------
['USER004', '', '5.00', '4']
Descrição:
1. Numero de elementos: 4
2. Contem o valor [USER]
3. O valor do PRODUTO esta VAZIO
4. PREÇO é um numero positivo
5. A QUANTIDADE é um numero inteiro


In [ ]:
def ler_arquivo(nome):
    for linha in open(nome, "r"):
        yield linha

vendas = ler_arquivo("Leitura.txt")

for venda in vendas:
    print(venda)

2024-01-15 08:30:00 INFO UsuÃ¡rio 123 fez login

2024-01-15 08:31:00 ERROR Falha na autenticaÃ§Ã£o do usuÃ¡rio 456

2024-01-15 08:32:00 WARNING Tentativa de acesso suspeita

2024-01-15 08:33:00 CRITICAL Servidor de banco de dados offline

2024-01-15 08:34:00 INFO Backup iniciado

2024-01-15 08:35:00 ERROR Timeout na conexÃ£o com API externa


In [2]:
#2º teste com yield
def exemplo_yield():
    for i in range(3):
        yield f"Item {i}" 

print("COM YIELD:")
for item in exemplo_yield():  # Precisa iterar!
    print("  ", item)

COM YIELD:
   Item 0
   Item 1
   Item 2


In [7]:
# PASSO A PASSO:
# 1. Use F1 (split) para dividir "USER123|stranger_things|3|2024-01-15"
# 2. Use I1 (validação múltipla) para validar cada campo:
#    - user_id deve começar com "USER"
#    - título não pode ser vazio
#    - episódio deve ser inteiro positivo (use F2)
#    - data deve ter 10 caracteres (formato básico)
# 3. Crie função processar_linha que retorna dicionário ou levanta erro
# 4. Use I2 (generator) para criar processar_logs que yield resultados válidos
# 5. Aplique I3 (operações) para limpar título: strip(), title()

# DICA FINAL: Comece criando:
def validar_netflix(user, titulo, episodio_str, data):
    """Versão específica Netflix da função I1"""
    erros = []
    # Implemente cada validação aqui
    return erros

logs = [
    "USER123|Filme1   |3|2024-01-15",
    "USER321| |6|2025-11-20",
    "uSER005|Filme3|8.0|2024-06-22",
]

NumElements = len(logs)
print(NumElements)

for LogPosicao in range(0, NumElements):
    #Separando as virgulas
    LogSeparado = logs[LogPosicao].strip() #passar por cada elemento
    LogSeparado = logs[LogPosicao].split('|')
    print(LogSeparado)

    #Verificar se o primeiro valor contem [USER]
    PrimeiroValor = LogSeparado[0]
    ContemUser = PrimeiroValor[0:4]
    
    if ContemUser == "USER":
        print("2. Contem o valor [USER]")
    else:
        print("2. Não contem [USER]")
    #=================================
    print(" ")
    SegundoValor = LogSeparado[1]
    SegundoValor = SegundoValor.strip()

    if SegundoValor == "":
        print("3. O valor do PRODUTO esta VAZIO")
    else:
        print("3. O valor do PRODUTO NÃO esta VAZIO")
#===========================================================
TerceiroValor = LogSeparado[2]
TerceiroValor = float(TerceiroValor)

if TerceiroValor.is_integer() and TerceiroValor > 0:
    print("É Inteiro e Positivo")
    if TerceiroValor > 0:
        print("4. É Inteiro e Positivo")
    else:
        print("4. É Inteiro e Negativo")
    
else:
    print("4. Não é Inteiro e é Negativo")

print(" ")


    
    
   

3
['USER123', 'Filme1   ', '3', '2024-01-15']
2. Contem o valor [USER]
 
3. O valor do PRODUTO NÃO esta VAZIO
['USER321', ' ', '6', '2025-11-20']
2. Contem o valor [USER]
 
3. O valor do PRODUTO esta VAZIO
['uSER005', 'Filme3', '8.0', '2024-06-22']
2. Não contem [USER]
 
3. O valor do PRODUTO NÃO esta VAZIO
É Inteiro e Positivo
4. É Inteiro e Positivo
 
